In [0]:
USE CATALOG meridian_t02;

-- 1. Quarantine Reconciliation: Bronze = Silver + Quarantine
SELECT 
    (SELECT COUNT(*) FROM bronze.bronze_reviews) AS bronze_reviews_count,
    (SELECT COUNT(*) FROM silver.silver_reviews) AS silver_reviews_count,
    (SELECT COUNT(*) FROM ops.quarantine_reviews) AS quarantine_reviews_count,
    ((SELECT COUNT(*) FROM silver.silver_reviews) + (SELECT COUNT(*) FROM ops.quarantine_reviews)) AS silver_plus_quarantine;

-- 2. Inspect Quarantined Rows and Captured Anomaly Reasons
SELECT quarantine_reason, COUNT(*) AS dropped_count
FROM ops.quarantine_reviews
GROUP BY quarantine_reason;

-- 3. SCD Type 2 Verification: Show historical versions of mutated user records
SELECT 
    user_id,
    user_type,
    country,
    __start_at,
    __end_at
FROM silver.dim_users
WHERE user_id IN (1, 2, 11, 12, 21, 22)
ORDER BY user_id, __start_at;

-- 4. Verify 30-Minute Inactivity Sessionisation
SELECT 
    session_id,
    user_id,
    COUNT(event) AS events_in_session,
    MIN(event_timestamp) AS session_start,
    MAX(event_timestamp) AS session_end,
    ROUND((UNIX_TIMESTAMP(MAX(event_timestamp)) - UNIX_TIMESTAMP(MIN(event_timestamp))) / 60.0, 2) AS duration_minutes
FROM silver.silver_clickstream
GROUP BY session_id, user_id
ORDER BY events_in_session DESC
LIMIT 10;


select * from gold.vdp_device_funnel;